In [0]:
from pyspark.sql.functions import col, row_number, sum, avg
from pyspark.sql.window import Window
from project_tools import merge_into_table

In [0]:
bills = spark.read.table("workspace.project_catalog.silver_layer")

In [0]:
sum_revenue = ( bills
                   .groupBy("bill_date")
                   .agg(sum("total_charge").alias("monthly_revenue"), avg("total_charge").alias("avg_revenue_per_room"))
                   )

In [0]:
merge_into_table(sum_revenue, ["bill_date"], "sum_revenue")

In [0]:
window = Window.partitionBy("room_id").orderBy(col("bill_date").desc())

latest_bills = (bills
                .withColumn("rn", row_number().over(window))
                .filter(col("rn") == 1)
                .drop("rn")
                )

In [0]:
merge_into_table(latest_bills, ["bill_id"], "gold_layer")